In [1]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 45.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=7b94c2b260e9905a0b3f7392652eafa0038bf809952da7701c767c9757d49dae
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Then change your output path to:
output_path = '/content/drive/MyDrive/chess_graph.json'

# And in the save step:
with open(output_path, "w") as f:
    json.dump(adjacency_list, f)

MessageError: Error: credential propagation was unsuccessful

In [2]:
import re
import chess
import chess.polyglot
from collections import defaultdict, Counter
import io
import time
import zstandard as zstd
from tqdm import tqdm

# Pre-compiled regex
HEADER_RE     = re.compile(r'\[(\w+)\s+"([^"]*)"\]')
MOVE_NUM_RE   = re.compile(r'\d+\.+')
RESULT_TOKENS = frozenset({'1-0', '0-1', '1/2-1/2', '*'})


# Fast PGN tokenizer
# the chess library does many additional things in the background, eg checking if a move is legal nor not,
# which costs alot of runtime, that is not relevant for parsing games, assuming no illegal moves were played and recorded in the database
def _strip_comments_and_variations(text: str) -> str:
    out   = []
    brace = 0
    paren = 0
    for ch in text:
        if   ch == '{': brace += 1
        elif ch == '}': brace = max(brace - 1, 0)
        elif brace == 0:
            if   ch == '(': paren += 1
            elif ch == ')': paren = max(paren - 1, 0)
            elif paren == 0:
                out.append(ch)
    return ''.join(out)


def _extract_san_moves(movetext: str) -> list:
    clean = _strip_comments_and_variations(movetext)
    clean = MOVE_NUM_RE.sub(' ', clean)
    return [t for t in clean.split() if t not in RESULT_TOKENS]


def _iter_raw_games(text_stream):
    headers    = []
    move_parts = []
    for raw in text_stream:
        line = raw.rstrip()
        if not line:
            continue
        if line[0] == '[':
            if move_parts:
                yield headers, ' '.join(move_parts)
                headers    = []
                move_parts = []
            headers.append(line)
        else:
            move_parts.append(line)
    if move_parts:
        yield headers, ' '.join(move_parts)


def _parse_headers(header_lines: list) -> dict:
    out = {}
    for line in header_lines:
        m = HEADER_RE.match(line)
        if m:
            out[m.group(1)] = m.group(2)
    return out


def _fmt_duration(seconds: float) -> str:
    """Format seconds into a readable string."""
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s:.1f}s"
    elif m > 0:
        return f"{m}m {s:.1f}s"
    else:
        return f"{s:.2f}s"


# Main pipeline

def parse_compressed_db(
    pgn_zst_path: str,
    max_nodes: int   = 1000,
    min_prob:  float = 0.05,
    min_count: int   = 100,
    max_depth: int   = 40,
) -> dict:

    hash_to_fen     = {}
    transitions     = defaultdict(Counter)
    outcomes        = defaultdict(lambda: [0, 0, 0])
    node_occurrence = Counter()

    dctx = zstd.ZstdDecompressor()

    game_count = processed_count = error_count = 0
    t_start    = time.perf_counter()

    print(f"Opening: {pgn_zst_path}\n")

    with open(pgn_zst_path, "rb") as fh, \
         dctx.stream_reader(fh) as reader:

        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')

        # tqdm: no `total` since exact game count is unknown without a pre-scan;
        # shows rate + elapsed + postfix stats instead
        with tqdm(
            _iter_raw_games(text_stream),
            desc="Pass 1 — parsing",
            unit=" games",
            miniters=500,
            bar_format="{desc}: {n_fmt} games [{elapsed}, {rate_fmt}]{postfix}",
        ) as pbar:

            for header_lines, movetext in pbar:
                game_count += 1

                # ── Filters ──────────────────────────────────────────────────
                headers = _parse_headers(header_lines)
                try:
                    w_elo     = int(headers.get("WhiteElo",    "0") or "0")
                    b_elo     = int(headers.get("BlackElo",    "0") or "0")
                    tc        = headers.get("TimeControl", "0+0")
                    main_time = int(tc.split('+')[0]) if '+' in tc else 0
                except (ValueError, IndexError):
                    continue

                if w_elo < 1000 or b_elo < 1000 or main_time < 300:
                    continue

                processed_count += 1
                res     = headers.get("Result", "*")
                res_idx = 0 if res == "1-0" else 1 if res == "0-1" else 2

                # Move processing, capped at max_depth
                san_moves = _extract_san_moves(movetext)
                board     = chess.Board()

                try:
                    for depth, san in enumerate(san_moves):
                        if depth >= max_depth:
                            break

                        curr_hash = chess.polyglot.zobrist_hash(board)

                        if curr_hash not in hash_to_fen:
                            hash_to_fen[curr_hash] = board.fen()

                        move = board.parse_san(san)
                        board.push(move)
                        next_hash = chess.polyglot.zobrist_hash(board)

                        transitions[curr_hash][next_hash] += 1
                        node_occurrence[curr_hash]         += 1
                        outcomes[curr_hash][res_idx]       += 1

                except (chess.InvalidMoveError, chess.IllegalMoveError,
                        chess.AmbiguousMoveError, ValueError):
                    error_count += 1
                    continue

                # Update postfix stats every 2000 games (low overhead)
                if game_count % 2000 == 0:
                    pbar.set_postfix(
                        kept   = f"{processed_count:,}",
                        errors = f"{error_count:,}",
                        nodes  = f"{len(node_occurrence):,}",
                    )

    t_pass1 = time.perf_counter()
    print(f"\n✓ Pass 1 done in {_fmt_duration(t_pass1 - t_start)}")
    print(f"  Scanned: {game_count:,} | Kept: {processed_count:,} | "
          f"Errors: {error_count:,} | Unique nodes: {len(node_occurrence):,}\n")

    # Pass 2: prune & build adjacency list
    top_nodes = {h for h, _ in node_occurrence.most_common(max_nodes)}
    final_adj = {}

    with tqdm(
        top_nodes,
        desc="Pass 2 — pruning",
        unit=" nodes",
        bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
    ) as pbar:
        for curr_hash in pbar:
            total_exits = sum(transitions[curr_hash].values())
            if total_exits < min_count:
                continue

            valid_edges = {
                nh: {"prob": round(cnt / total_exits, 4), "count": cnt}
                for nh, cnt in transitions[curr_hash].items()
                if cnt / total_exits >= min_prob and nh in top_nodes
            }

            if valid_edges:
                final_adj[curr_hash] = {
                    "edges":        valid_edges,
                    "outcomes":     outcomes[curr_hash],
                    "total_visits": total_exits,
                    "fen":          hash_to_fen.get(curr_hash),
                }

    t_end = time.perf_counter()
    print(f"\n✓ Pass 2 done in {_fmt_duration(t_end - t_pass1)}")
    print(f"  Final graph: {len(final_adj)} nodes\n")
    print(f"{'─'*40}")
    print(f"  Total runtime: {_fmt_duration(t_end - t_start)}")
    print(f"{'─'*40}\n")

    return final_adj


In [3]:
# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import json

    MY_FILE_PATH = '/content/lichess_db_standard_rated_2013-01.pgn.zst'

    adjacency_list = parse_compressed_db(MY_FILE_PATH)

    print("Saving chess_graph.json ...")
    with open("chess_graph.json", "w") as f:
        json.dump(adjacency_list, f)
    print("Done!")

Opening: /content/lichess_db_standard_rated_2013-01.pgn.zst



Pass 1 — parsing: 121332 games [05:11, 389.18 games/s], errors=168, kept=59,409, nodes=1,672,687



✓ Pass 1 done in 5m 11.8s
  Scanned: 121,332 | Kept: 60,092 | Errors: 174 | Unique nodes: 1,691,276



Pass 2 — pruning: 100%|██████████| 1000/1000 [00:00]



✓ Pass 2 done in 0.35s
  Final graph: 422 nodes

────────────────────────────────────────
  Total runtime: 5m 12.1s
────────────────────────────────────────

Saving chess_graph.json ...
Done!
